# Inner Speech EEG Decoding and Spectral Connectivity
Run the EEG pipeline for one participant, then inspect paired-trial QC, alpha connectivity and multitaper power. If both sessions meet the class-count requirement, the pipeline also reports held-out-session decoding. Install the repository in the notebook kernel and download the full BrainVision recordings before running. See the [execution guide](../../docs/ds006033.md).

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file())
dataset = root / "data" / "ds006033"  # Edit for a different download location
subject = "sub-02"
from inner_speech.ds006033.eeg import paired_events

output = root / "results" / "generated" / "ds006033" / "notebook-eeg"
session = "ses-01"  # Session to inspect below
peak_uv = 200  # Default paired QC threshold
scratch = None  # Optional Path on a drive with several GB free

In [ ]:
prefix = f"{subject}_{session}_task-innerspeech"
eeg_dir = dataset / subject / session / "eeg"
events = eeg_dir / f"{prefix}_events.tsv"
pairs = paired_events(events)
display(pairs.head())
display(pairs["word"].value_counts().rename("candidate pairs"))

### Run the pipeline
The command processes all available EEG sessions for the participant. If QC leaves no pairs, the command stops. Review rejected segments and residual MRI artifacts before changing the threshold.

In [ ]:
command = [
    sys.executable, "-m", "inner_speech.ds006033.eeg",
    str(dataset), str(output), "--subject", subject,
    "--reject-peak-to-peak-uv", str(peak_uv),
]
if scratch is not None:
    command.extend(["--scratch", str(scratch)])
subprocess.run(command, check=True)

In [ ]:
import numpy as np

session_output = output / subject / session
display(json.loads((session_output / "qc.json").read_text()))
display(pd.read_csv(session_output / "rejected.csv").head())

for method in ("lda", "svm"):
    scores = output / subject / f"csp_{method}_folds.csv"
    if scores.is_file():
        display(pd.read_csv(scores))

connectivity = session_output / "connectivity.npz"
if connectivity.is_file():
    with np.load(connectivity) as values:
        display(pd.DataFrame({
            "method": values["methods"],
            "mean task": values["task"][:, :, 0].mean(axis=1),
            "mean inter-trial rest": values["rest"][:, :, 0].mean(axis=1),
        }))
else:
    print("No connectivity output for this session. See the paired-trial count above.")

In [ ]:
with np.load(session_output / "task_power.npz") as values:
    power = values["power"].mean(axis=0)
    fig, ax = plt.subplots()
    mesh = ax.pcolormesh(values["times"], values["freqs"], power, shading="auto")
    ax.set(xlabel="Seconds from analysis-window start", ylabel="Frequency (Hz)",
           title="Mean sensor power during the word window")
    fig.colorbar(mesh, ax=ax, label="Power (V²)")
    plt.show()